# JRA-3Q 気圧配置JSON変換（Colab版）

手元のPCでの `git` 操作がうまくいかない場合向けに、**Google Drive上の
.ncファイルを読み込み → 台風ごとのJSONに変換 → GitHubへ直接push**まで
Colab上で完結させるノートブックです。

## 前提
- ダウンロード済みの（landfallJP+damaging分の） `.nc` ファイル（`jra3q_colab_download.ipynb` で
  取得したもの）が **Google Driveのどこかのフォルダに置いてある**こと
- GitHubへpushする権限（Personal Access Token）を用意すること
  （下の②で作り方を説明します）

## 使い方
①→②→③→④→⑤の順に上から実行してください。

## ① Google Driveをマウントし、.ncファイルのフォルダを指定

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Google Drive内で .nc ファイルが入っているフォルダのパスに書き換えてください。
# 例: '/content/drive/MyDrive/jra3q_pressure_japan'
RAW_DIR = '/content/drive/MyDrive/jra3q_pressure_japan'

import pathlib
n = len(list(pathlib.Path(RAW_DIR).glob('*.nc')))
print(f'{RAW_DIR} 内に .nc ファイル {n} 個')

## ② GitHubへのPersonal Access Token（PAT）を用意

すでに持っていれば③に進んでOKです。まだの場合:

1. GitHubの https://github.com/settings/tokens?type=beta を開く
2. **Generate new token** → このリポジトリ（`awg-yk/typhoon-wind-rainfall`）
   に対して **Contents: Read and write** 権限を付与
3. 発行されたトークン（`github_pat_...` から始まる文字列）をコピー

次のセルを実行すると入力欄が出るので、そこに貼り付けてください
（画面には表示されず、Colab上にも保存されません）。

In [ ]:
import getpass

GITHUB_TOKEN = getpass.getpass('GitHubのPersonal Access Tokenを貼り付けてEnter: ')

## ③ リポジトリを取得し、必要なライブラリをインストール

In [ ]:
import os

REPO_DIR = '/content/typhoon-wind-rainfall'
BRANCH = 'claude/remaining-tasks-gzid2a'
REMOTE_URL = f'https://{GITHUB_TOKEN}@github.com/awg-yk/typhoon-wind-rainfall.git'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REMOTE_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}

!cd {REPO_DIR} && git config user.email "colab@example.com"
!cd {REPO_DIR} && git config user.name "Colab"

!pip install -q xarray netCDF4

## ④ 変換実行

`data/raw_jra3q/*.nc`（landfallJP+damaging分）を台風ごとの経路期間に合わせて
`data/pressure/<台風コード>.json` に切り出します。該当月のファイルが
Drive内に無い台風は `FAILED (missing raw file(s) for ...)` と表示されて
スキップされます（想定内の挙動です）。

In [ ]:
!cd {REPO_DIR} && python3 scripts/build_pressure_json.py --raw-dir "{RAW_DIR}"

import pathlib
out_dir = pathlib.Path(REPO_DIR) / 'data' / 'pressure'
files = list(out_dir.glob('*.json'))
total_mb = sum(f.stat().st_size for f in files) / 1e6
print(f'{len(files)} JSON files, {total_mb:.1f} MB total')

## ⑤ GitHubへコミット & push

`data/pressure/` 配下のJSONだけをコミットします
（`git status` の出力で他のファイルが混ざっていないか一応確認してください）。

In [ ]:
!cd {REPO_DIR} && git add data/pressure/
!cd {REPO_DIR} && git status --short
!cd {REPO_DIR} && git commit -m "Add JRA-3Q pressure data (data/pressure/*.json)"
!cd {REPO_DIR} && git push origin {BRANCH}

## 完了後

- GitHub上の該当ブランチに `data/pressure/*.json` が反映されていれば成功です。
- このノートブックのセッションを閉じれば、貼り付けたトークンはColab上から
  消えます（`GITHUB_TOKEN` 変数はこのランタイムのメモリ上にのみ存在します）。
  念のため、使い終わったトークンはGitHubの設定画面から失効させておくと
  より安全です。